In [24]:
import pandas as pd
from nltk.tokenize import word_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
import string
import numpy as np
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout, Input
from tensorflow.keras.utils import to_categorical
from random import randint

In [2]:
df = pd.read_csv("output/raw.csv")

In [11]:
# text = ''.join([c for c in ' '.join(df.dropna().values.flatten()).lower() if c not in string.punctuation])
text = ' '.join([sentence.strip() if sentence.strip()[-1] == '.' else sentence.strip() + '.' for sentence in df.dropna().values.flatten()]).lower()
words = word_tokenize(text)
n_words = len(words)
unique_words = len(set(words))

print('Total Words: %d' % n_words)
print('Unique Words: %d' % unique_words)

Total Words: 65523
Unique Words: 8341


In [12]:
unique_chars = sorted(list(set(text)))
char_to_index = {char: idx for idx, char in enumerate(unique_chars)}
char_to_index

{' ': 0,
 '!': 1,
 '&': 2,
 "'": 3,
 '(': 4,
 ')': 5,
 '+': 6,
 ',': 7,
 '-': 8,
 '.': 9,
 '/': 10,
 '0': 11,
 '1': 12,
 '2': 13,
 '3': 14,
 '4': 15,
 '5': 16,
 '6': 17,
 '7': 18,
 '8': 19,
 '9': 20,
 ':': 21,
 ';': 22,
 '[': 23,
 '\\': 24,
 ']': 25,
 '_': 26,
 'a': 27,
 'b': 28,
 'c': 29,
 'd': 30,
 'e': 31,
 'f': 32,
 'g': 33,
 'h': 34,
 'i': 35,
 'j': 36,
 'k': 37,
 'l': 38,
 'm': 39,
 'n': 40,
 'o': 41,
 'p': 42,
 'q': 43,
 'r': 44,
 's': 45,
 't': 46,
 'u': 47,
 'v': 48,
 'w': 49,
 'x': 50,
 'y': 51,
 'z': 52,
 '|': 53,
 '~': 54,
 '\xa0': 55,
 '´': 56,
 'á': 57,
 'é': 58,
 'ó': 59,
 'ł': 60,
 'ń': 61,
 'œ': 62,
 '’': 63,
 '“': 64,
 '”': 65,
 '„': 66,
 '！': 67,
 '，': 68,
 '🌀': 69,
 '🐴': 70,
 '👜': 71,
 '👝': 72,
 '🙉': 73,
 '🤔': 74,
 '🤨': 75,
 '🥮': 76,
 '🧐': 77,
 '🫙': 78}

In [13]:
input_sequence = []
output_words = []
input_seq_length = 40

for i in range(0, len(text) - input_seq_length , 1):
    in_seq = text[i:i + input_seq_length]
    out_seq = text[i + input_seq_length]
    input_sequence.append([char_to_index[word] for word in in_seq])
    output_words.append(char_to_index[out_seq])

In [15]:
X = np.reshape(input_sequence, (len(input_sequence), input_seq_length, 1))
X = X / float(len(unique_chars))

y = to_categorical(output_words, num_classes=len(unique_chars))

In [16]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (338510, 40, 1)
y shape: (338510, 79)


In [25]:
model = Sequential([
    Input((input_seq_length, 1)),
    LSTM(128, return_sequences=True),
    LSTM(128),
    Dense(len(unique_chars), activation='softmax')
])
model.summary()

model.compile(loss='categorical_crossentropy', optimizer='adam')

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                        │ (None, 40, 128)             │          66,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 79)                  │          10,191 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 208,335 (813.81 KB)

 Trainable params: 208,335 (813.81 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.fit(X, y, batch_size=1024, epochs=10, verbose=1)

Epoch 1/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 109s 323ms/step - loss: 3.1677
Epoch 2/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 106s 320ms/step - loss: 3.0088
Epoch 3/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 104s 314ms/step - loss: 2.9824
Epoch 4/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 103s 312ms/step - loss: 2.8758
Epoch 5/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 103s 310ms/step - loss: 2.8011
Epoch 6/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 102s 309ms/step - loss: 2.7290
Epoch 7/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 103s 310ms/step - loss: 2.6691
Epoch 8/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 103s 310ms/step - loss: 2.6204
Epoch 9/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 103s 312ms/step - loss: 2.5708
Epoch 10/10
331/331 ━━━━━━━━━━━━━━━━━━━━ 103s 310ms/step - loss: 2.5262


In [19]:
import random

# Pick a random starting sequence
start_index = random.randint(0, len(text) - input_seq_length - 1)
seed_text = text[start_index:start_index + input_seq_length]

# Generate characters
generated_text = seed_text
for _ in range(200):  # Generate 200 characters
    input_seq = np.array([[char_to_index[char] for char in seed_text]]).reshape(1, input_seq_length, 1)
    input_seq = input_seq / float(len(unique_chars))

    # Predict the next character
    predicted_index = np.argmax(model.predict(input_seq, verbose=0))
    next_char = unique_chars[predicted_index]

    # Append to generated text and update seed
    generated_text += next_char
    seed_text = seed_text[1:] + next_char

print(generated_text)

sa, snake hair, naughty smile, wearing and artse sith and a cere and a bare and a bare and a bare and a bare and a bare and a bare and a bare and a bare and a bare and a bare and a bare and a bare and a bare and a bare and a bare and a bare


In [38]:
from transformers import pipeline

# Load a pre-trained model for text classification
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

# Define the sentence and candidate labels (themes)
sentence = "The stock market crashed due to unexpected economic downturn."
candidate_labels = ["finance", "sports", "politics", "technology", "health"]

# Perform the classification
result = classifier(sentence, candidate_labels)

# Print the result
print(f"Sentence: {sentence}")
print(f"Detected Theme: {result['labels'][0]} with confidence {result['scores'][0]:.2f}")

Device set to use cpu


Sentence: The stock market crashed due to unexpected economic downturn.
Detected Theme: finance with confidence 0.95
